In [ ]:
# --- Скачивание ---
!python src/download_data.py --config configs/config.yaml

# ===== ЭТАП 1: RAW =====
!python src/analyze_annotations.py --stage raw
!python src/sample_and_draw.py --stage raw --num 30 --grid
# смотрим data/reports/raw/*.png и data/samples/raw/contact_sheet.jpg
# (тут ожидаемо МОГУТ быть проблемы — это «золотой» источник)

# --- Предобработка + фильтрация ---
!python src/preprocess.py --config configs/config.yaml
# в логе будет сводка: сколько боксов оставлено/отвергнуто/удалено изображений

# ===== ЭТАП 2: STAGED =====
!python src/analyze_annotations.py --stage staged
!python src/sample_and_draw.py --stage staged --num 50 --grid
# смотрим data/reports/staged/*.png и data/samples/staged/contact_sheet.jpg
# здесь уже должно быть ЧИСТО: out_of_bounds=0, area_median в норме
# дополнительно: посмотреть data/staged/rejected_annotations.csv — что и почему удалили

# --- Разделение и YOLO ---
!python src/split_data.py --config configs/config.yaml

# ===== ЭТАП 3: PROCESSED =====
!python src/analyze_annotations.py --stage processed --split train
!python src/analyze_annotations.py --stage processed --split val
!python src/sample_and_draw.py --stage processed --split train --num 50 --grid
!python src/sample_and_draw.py --stage processed --split val   --num 50 --grid

# --- ГЛАВНАЯ проверка: raw/staged vs processed ---
!python src/compare_stages.py --num 40 --mode overlay --split train
!python src/compare_stages.py --num 40 --mode side    --split val
# красные боксы (из staged) должны точно совпасть с зелёными (YOLO)

# --- Обучение ---
!python src/train.py --config configs/config.yaml
!python src/evaluate.py --config configs/config.yaml